# T5Gemma2 greedy drafter: PubMed smoke pilot on Colab L4

This notebook runs the current `src/encdec_drafter` implementation through Google Colab CLI. It uses the **base** `google/t5gemma-2-1b-1b` checkpoint as a frozen teacher, downloads the weights only to the Colab VM, and uses small train/validation slices from `ccdv/pubmed-summarization` to check the end-to-end pipeline.

This is a systems smoke test, not a paper result: 8 training articles, 4 validation articles, 4,096 source-token cap, 64 generated tokens, 32 drafter updates, greedy decoding, and exact token comparison against native cached generation. The full fine-tuned checkpoint used by the paper is not in this local workspace, so do not interpret this base-checkpoint pilot as its performance. The test split is not touched.

The CLI runner uploads a temporary package archive and a short-lived HF token file to `/content`. The token cell deletes its remote file before model download. When opened and run from the Colab UI instead, set a Colab Secret named `HF_TOKEN` and upload the package archive. The existing prototype notebook is left unchanged.


In [ ]:
# @title 1. Runtime and pilot limits
import json
import os
import random
import shutil
import subprocess
import sys
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import torch

RUNTIME_READY = PACKAGE_READY = MODEL_READY = DATA_READY = CONFIG_READY = TRAIN_READY = False

SEED = 42
TARGET_MODEL_ID = "google/t5gemma-2-1b-1b"
DATASET_ID = "ccdv/pubmed-summarization"
DATASET_CONFIG = "document"
TRAIN_LIMIT = 8
VALIDATION_LIMIT = 4
MAX_SOURCE_TOKENS = 4096
MAX_NEW_TOKENS = 64
TRAIN_STEPS = 32
BLOCK_SIZE = 4
BENCHMARK_REPEATS = 2

if not torch.cuda.is_available():
    raise RuntimeError("Choose a Colab GPU runtime before running this notebook.")
GPU_NAME = torch.cuda.get_device_name(0)
if "L4" not in GPU_NAME:
    raise RuntimeError(f"This pilot is configured for L4, but Colab assigned {GPU_NAME!r}.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The L4 runtime does not report BF16 support; stop rather than silently changing precision.")

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

RUN_ROOT = Path("/content/t5gemma2_drafter_colab")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RUN_DIR = RUN_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
RUNTIME_READY = True
HF_HOME = Path("/content/t5gemma2_model_cache")
os.environ["HF_HOME"] = str(HF_HOME)

print(
    json.dumps(
        {
            "gpu": GPU_NAME,
            "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
            "torch": torch.__version__,
            "bf16": True,
            "run_dir": str(RUN_DIR),
            "limits": {
                "train": TRAIN_LIMIT,
                "validation": VALIDATION_LIMIT,
                "source_tokens": MAX_SOURCE_TOKENS,
                "max_new_tokens": MAX_NEW_TOKENS,
                "train_steps": TRAIN_STEPS,
            },
        },
        indent=2,
    )
)

In [ ]:
# @title 2. Stage the repository code and matching runtime dependency
import importlib.metadata
import zipfile

if not RUNTIME_READY:
    raise RuntimeError("Runtime setup did not complete; refusing to use stale kernel state.")

PACKAGE_ARCHIVE = Path("/content/encdec_drafter_src.zip")
if not PACKAGE_ARCHIVE.is_file():
    raise FileNotFoundError(
        "Upload src/encdec_drafter/encdec_drafter as /content/encdec_drafter_src.zip with colab upload before running."
    )
PACKAGE_ROOT = RUN_DIR / "package"
PACKAGE_ROOT.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(PACKAGE_ARCHIVE) as archive:
    archive.extractall(PACKAGE_ROOT)
sys.path[:] = [entry for entry in sys.path if "/t5gemma2_drafter_colab/" not in entry]
for module_name in tuple(sys.modules):
    if module_name == "encdec_drafter" or module_name.startswith("encdec_drafter."):
        del sys.modules[module_name]
sys.path.insert(0, str(PACKAGE_ROOT))
importlib.invalidate_caches()

TRANSFORMERS_VERSION = "5.14.1"
TOKENIZERS_VERSION = "0.22.2"


def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


if installed_version("transformers") != TRANSFORMERS_VERSION or installed_version("tokenizers") != TOKENIZERS_VERSION:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-deps",
            f"transformers=={TRANSFORMERS_VERSION}",
            f"tokenizers=={TOKENIZERS_VERSION}",
        ],
        check=True,
    )

import yaml
from encdec_drafter.config import load_config

print("transformers", importlib.metadata.version("transformers"))
print("tokenizers", importlib.metadata.version("tokenizers"))
print("package root", PACKAGE_ROOT)
PACKAGE_READY = True
print("Package archive stays on the Colab VM; no target model is copied to the Mac.")

In [ ]:
# @title 3. Consume the short-lived HF token and fetch weights on Colab only
from huggingface_hub import snapshot_download

if not PACKAGE_READY:
    raise RuntimeError("Source/dependency staging did not complete; refusing to use stale kernel state.")

TOKEN_FILE = Path("/content/.hf_token_runtime")
if TOKEN_FILE.is_file():
    hf_token = TOKEN_FILE.read_text(encoding="utf-8").strip()
    TOKEN_FILE.unlink()
else:
    try:
        from google.colab import userdata

        hf_token = userdata.get("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "HF access is needed for this gated checkpoint. For CLI runs, upload the one-use token file; for Colab UI runs, add an HF_TOKEN Secret."
        ) from exc
if not hf_token:
    raise RuntimeError("No HF_TOKEN was supplied; no model weights were downloaded.")

MODEL_PATH = Path(
    snapshot_download(
        repo_id=TARGET_MODEL_ID,
        token=hf_token,
        cache_dir=str(HF_HOME / "hub"),
    )
).resolve()
MODEL_READY = True
del hf_token
print("Model cached on Colab:", MODEL_PATH)
print("Model revision:", MODEL_PATH.name)
print("HF token removed from the runtime variable and temporary file deleted.")

In [ ]:
# @title 4. Fetch a deterministic train/validation slice from the public PubMed dataset
import requests

if not MODEL_READY:
    raise RuntimeError("Model staging did not complete; refusing to use stale kernel state.")

DATA_DIR = RUN_DIR / "data"
DATA_DIR.mkdir()


def fetch_dataset_rows(split, offset, length):
    params = {"dataset": DATASET_ID, "config": DATASET_CONFIG, "split": split, "offset": offset, "length": length}
    response = None
    for attempt in range(6):
        try:
            response = requests.get("https://datasets-server.huggingface.co/rows", params=params, timeout=30)
            if response.status_code not in {429, 500, 502, 503, 504}:
                response.raise_for_status()
                break
            if attempt == 5:
                response.raise_for_status()
        except requests.RequestException:
            if attempt == 5:
                raise
        wait_seconds = min(30, 2**attempt)
        print(f"Dataset API transient failure for {split}; retrying in {wait_seconds}s ({attempt + 1}/6).", flush=True)
        time.sleep(wait_seconds)
    if response is None:
        raise RuntimeError(f"Dataset API did not return a response for {split}.")
    response.raise_for_status()
    records = [item["row"] for item in response.json()["rows"]]
    if not records:
        raise RuntimeError(f"No rows returned for {split}.")
    return records


def fetch_usable_rows(split, limit):
    selected = []
    for offset in range(0, 64, 2):
        page = fetch_dataset_rows(split, offset, 2)
        selected.extend(
            row
            for row in page
            if isinstance(row.get("article"), str)
            and row["article"].strip()
            and isinstance(row.get("abstract"), str)
            and row["abstract"].strip()
        )
        if len(selected) >= limit:
            return selected[:limit]
    raise RuntimeError(f"Only {len(selected)} usable rows returned for {split}; expected {limit}.")


train_rows = fetch_usable_rows("train", TRAIN_LIMIT)
validation_rows = fetch_usable_rows("validation", VALIDATION_LIMIT)

TRAIN_FILE = DATA_DIR / "train.jsonl"
VALIDATION_FILE = DATA_DIR / "validation.jsonl"


def save_split(path, rows, split):
    with path.open("w", encoding="utf-8") as handle:
        for index, row in enumerate(rows):
            output = {
                "id": str(row.get("id") or f"{split}-{index}"),
                "text": row["article"],
                "summary": row["abstract"],
            }
            handle.write(json.dumps(output, ensure_ascii=False) + "\n")


save_split(TRAIN_FILE, train_rows, "train")
save_split(VALIDATION_FILE, validation_rows, "validation")
DATA_READY = True
print(
    json.dumps(
        {
            "dataset": DATASET_ID,
            "config": DATASET_CONFIG,
            "train_examples": len(train_rows),
            "validation_examples": len(validation_rows),
            "training_ids": [str(row.get("id", "")) for row in train_rows],
            "validation_ids": [str(row.get("id", "")) for row in validation_rows],
            "test_split_used": False,
        },
        indent=2,
    )
)

In [ ]:
# @title 5. Write a local-only runtime config matching PubMed prompts and decoding

if not DATA_READY:
    raise RuntimeError("Dataset staging did not complete; refusing to use stale kernel state.")

SOURCE_PREFIX = (
    "Summarize the following biomedical research article into a concise, factual abstract. "
    "Preserve the key objective, methods, results, and conclusion; do not add information.\nArticle:\n"
)
TRACE_FILE = RUN_DIR / "teacher_traces.jsonl"
DRAFTER_FILE = RUN_DIR / "t5gemma2_drafter.pt"
BENCH_DIR = RUN_DIR / "benchmark"
CONFIG_PATH = RUN_DIR / "runtime.yaml"
config = {
    "target": {
        "checkpoint": str(MODEL_PATH),
        "identity": f"{TARGET_MODEL_ID}@{MODEL_PATH.name}",
        "tokenizer": None,
        "local_files_only": True,
        "trust_remote_code": False,
        "device": "cuda",
        "dtype": "bfloat16",
        "attn_implementation": None,
    },
    "data": {
        "training_file": str(TRAIN_FILE),
        "validation_file": str(VALIDATION_FILE),
        "test_file": str(VALIDATION_FILE),
        "id_field": "id",
        "source_field": "text",
        "target_field": "summary",
        "detokenize": True,
        "source_prefix": SOURCE_PREFIX,
        "max_source_tokens": MAX_SOURCE_TOKENS,
    },
    "generation": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "min_new_tokens": 32,
        "repetition_penalty": 1.05,
        "no_repeat_ngram_size": 3,
        "temperature": 0.0,
        "top_k": 0,
        "top_p": 1.0,
        "num_beams": 1,
    },
    "draft": {
        "width": 256,
        "layers": 2,
        "heads": 4,
        "ffn_width": 1024,
        "max_block_size": 8,
        "target_feature_window": 64,
        "max_source_tokens": MAX_SOURCE_TOKENS,
        "checkpoint": None,
    },
    "source_memory": {"mode": "full", "budget": 256},
    "speculation": {"block_size": BLOCK_SIZE},
    "training": {
        "trace_file": str(TRACE_FILE),
        "output_checkpoint": str(DRAFTER_FILE),
        "steps": TRAIN_STEPS,
        "learning_rate": 0.0003,
        "weight_decay": 0.01,
        "seed": SEED,
        "log_every": 8,
        "save_every": 32,
        "max_grad_norm": 1.0,
    },
    "benchmark": {"output_dir": str(BENCH_DIR), "warmup": 1, "repeats": BENCHMARK_REPEATS},
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
loaded_config = load_config(CONFIG_PATH)
CONFIG_READY = True
print("config", CONFIG_PATH)
print("model loading is local-only after Colab cache download")
print("decoder policy: greedy; max_new_tokens=64; min_new_tokens=32; repetition=1.05; no-repeat trigram")

In [ ]:
# @title 6. Load the target, profile, collect teacher traces, and train the drafter
import gc

from encdec_drafter.benchmark import profile_t5gemma2
from encdec_drafter.data import collect_teacher_traces, read_jsonl
from encdec_drafter.model import load_adapter
from encdec_drafter.training import train_drafter

if not CONFIG_READY:
    raise RuntimeError("Configuration staging did not complete; refusing to use stale kernel state.")

start_time = time.perf_counter()
adapter = load_adapter(loaded_config)
print("target", json.dumps(adapter.target_manifest(), ensure_ascii=False), flush=True)

PROFILE_FILE = RUN_DIR / "profile.json"
profile_result = profile_t5gemma2(adapter, loaded_config, next(read_jsonl(VALIDATION_FILE, limit=1)))
PROFILE_FILE.write_text(json.dumps(profile_result, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("profile", json.dumps(profile_result, ensure_ascii=False), flush=True)
del profile_result
gc.collect()
torch.cuda.empty_cache()

trace_result = collect_teacher_traces(adapter, loaded_config, TRAIN_FILE, TRACE_FILE, limit=TRAIN_LIMIT, overwrite=True)
print("teacher_traces", json.dumps(trace_result, ensure_ascii=False), flush=True)

training_result = train_drafter(adapter, loaded_config, TRACE_FILE, output_path=DRAFTER_FILE, max_steps=TRAIN_STEPS)
print("training", json.dumps(training_result, ensure_ascii=False), flush=True)
TRAIN_READY = True

In [ ]:
# @title 7. Paired native/speculative validation and decision gate
from encdec_drafter.benchmark import benchmark_runtime
from encdec_drafter.data import source_text
from encdec_drafter.generation import GreedySpeculativeGenerator

if not TRAIN_READY:
    raise RuntimeError("Drafter training did not complete; refusing to use stale kernel state.")

# Isolate target/cache/policy correctness from drafter proposal quality.
diagnostic_row = next(read_jsonl(VALIDATION_FILE, limit=1))
diagnostic_ids, diagnostic_mask = adapter.tokenize_source(source_text(diagnostic_row, loaded_config))
native_diagnostic = [
    int(value) for value in adapter.native_generate(diagnostic_ids, diagnostic_mask).detach().cpu().tolist()
]
diagnostic_source = adapter.encode_ids(diagnostic_ids, diagnostic_mask)
target_only_diagnostic = (
    GreedySpeculativeGenerator(adapter, loaded_config).generate_source(diagnostic_source).decoder_ids
)
first_target_only_mismatch = next(
    (i for i, (left, right) in enumerate(zip(native_diagnostic, target_only_diagnostic)) if left != right), None
)
if first_target_only_mismatch is None and len(native_diagnostic) != len(target_only_diagnostic):
    first_target_only_mismatch = min(len(native_diagnostic), len(target_only_diagnostic))
target_only_parity = {
    "exact": native_diagnostic == target_only_diagnostic,
    "native_length": len(native_diagnostic),
    "target_only_length": len(target_only_diagnostic),
    "first_mismatch_position": first_target_only_mismatch,
    "native_token": native_diagnostic[first_target_only_mismatch]
    if first_target_only_mismatch is not None and first_target_only_mismatch < len(native_diagnostic)
    else None,
    "target_only_token": target_only_diagnostic[first_target_only_mismatch]
    if first_target_only_mismatch is not None and first_target_only_mismatch < len(target_only_diagnostic)
    else None,
    "native_neighborhood": native_diagnostic[max(0, first_target_only_mismatch - 3) : first_target_only_mismatch + 4]
    if first_target_only_mismatch is not None
    else [],
    "target_only_neighborhood": target_only_diagnostic[
        max(0, first_target_only_mismatch - 3) : first_target_only_mismatch + 4
    ]
    if first_target_only_mismatch is not None
    else [],
}
print("target_only_parity", json.dumps(target_only_parity, ensure_ascii=False), flush=True)
del diagnostic_source

benchmark_result = benchmark_runtime(
    adapter,
    loaded_config,
    VALIDATION_FILE,
    limit=VALIDATION_LIMIT,
    repeats=BENCHMARK_REPEATS,
    warmup=1,
    draft_checkpoint=DRAFTER_FILE,
    output_dir=BENCH_DIR,
)
benchmark_summary = json.loads((Path(benchmark_result["run_dir"]) / "summary.json").read_text(encoding="utf-8"))
exact_rate = float(benchmark_summary["exact_token_match_rate"])
median_speedup = float(benchmark_summary["per_request_speedup"]["median"])
if exact_rate < 1.0:
    decision = "STOP: exact greedy parity failed; do not interpret speed numbers."
elif median_speedup <= 1.0:
    decision = "STOP: this pilot did not show a latency win."
else:
    decision = "Positive smoke signal only; confirm on a larger validation slice before making a speed claim."

pilot_summary = {
    "status": "smoke_pilot_completed",
    "target_model": TARGET_MODEL_ID,
    "target_checkpoint_type": "base checkpoint, not the paper's PubMed fine-tuned checkpoint",
    "target_revision": MODEL_PATH.name,
    "gpu": GPU_NAME,
    "dataset": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "train_examples": TRAIN_LIMIT,
    "validation_examples": VALIDATION_LIMIT,
    "source_token_cap": MAX_SOURCE_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "drafter_steps": TRAIN_STEPS,
    "drafter_mean_loss": training_result["mean_loss"],
    "block_size": BLOCK_SIZE,
    "target_only_parity": target_only_parity,
    "exact_token_match_rate": exact_rate,
    "median_per_request_speedup": median_speedup,
    "native_peak_allocated_bytes": benchmark_summary.get("native_peak_allocated_bytes"),
    "speculative_peak_allocated_bytes": benchmark_summary.get("speculative_peak_allocated_bytes"),
    "decision": decision,
    "benchmark_summary": str(Path(benchmark_result["run_dir"]) / "summary.json"),
    "drafter_checkpoint": str(DRAFTER_FILE),
    "elapsed_seconds": round(time.perf_counter() - start_time, 2),
    "not_a_paper_result": True,
}
SUMMARY_FILE = RUN_DIR / "pilot_summary.json"
SUMMARY_FILE.write_text(json.dumps(pilot_summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
RESULTS_BUNDLE = RUN_DIR / "colab_results.zip"
with zipfile.ZipFile(RESULTS_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact in (
        SUMMARY_FILE,
        PROFILE_FILE,
        Path(benchmark_result["run_dir"]) / "summary.json",
        Path(benchmark_result["run_dir"]) / "requests.jsonl",
        TRAIN_FILE,
        VALIDATION_FILE,
    ):
        archive.write(artifact, artifact.relative_to(RUN_DIR))
print(json.dumps(pilot_summary, indent=2, ensure_ascii=False))
print("PILOT_SUMMARY_PATH=" + str(SUMMARY_FILE))
print("RESULTS_BUNDLE_PATH=" + str(RESULTS_BUNDLE))

## Interpretation

Use exact token parity as a gate: if it is below 100%, treat the speculative runtime as incorrect and do not use its latency numbers. If parity is exact but median speedup is at most 1.0, this implementation has not accelerated this workload. Even a positive result here is only an early smoke signal: the sample is tiny, uses the base checkpoint, limits generation to 64 tokens, and does not represent the full PubMed test evaluation.
